In [560]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
import PIL.Image as Image
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

**We will treat different sides of the iris as different subjects , This is a check for the balance of the data classes , and we can see that the data is reasonably balanced**

In [561]:
data_path = 'DataSet/CASIA-Iris-Thousand' 
folders = os.listdir(data_path)
total_classes = 0
image_counts = {}
for fold in folders:
    sub_path = os.path.join(data_path, fold)
    if os.path.isdir(sub_path):
        for side in ['L', 'R']:
            side_path = os.path.join(sub_path, side)
            if os.path.exists(side_path):
                total_classes += 1
                imgs = os.listdir(side_path)
                image_counts[side_path] = len(imgs)
print(f"Total Unique Irises (Classes): {total_classes}")
print(f"Average Images per Class: {sum(image_counts.values())/len(image_counts):.2f}")
print(f"Min images: {min(image_counts.values())}, Max images: {max(image_counts.values())}")

Total Unique Irises (Classes): 2000
Average Images per Class: 10.01
Min images: 10, Max images: 11


In [562]:
label_mapping=pd.read_csv(r"DataSet\iris_thousands.csv",index_col=0)

In [563]:
label_mapping

,Label,ImagePath
0,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
1,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
2,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
3,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
4,437-R,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
...,...,...
19995,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
19996,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
19997,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...
19998,715-L,/kaggle/input/casia-iris-thousand/CASIA-Iris-T...


In [564]:
label_mapping['ImagePath'] = label_mapping['ImagePath'].str.replace(r'^/kaggle/input/casia-iris-thousand/','DataSet/',regex=True)

In [565]:
label_mapping

,Label,ImagePath
0,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R06.jpg
1,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R09.jpg
2,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R07.jpg
3,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R02.jpg
4,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R03.jpg
...,...,...
19995,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L04.jpg
19996,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L09.jpg
19997,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L08.jpg
19998,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L06.jpg


**Now we need to label encode the classes to be able to use them in the model training**

In [566]:
le = LabelEncoder()
label_mapping['encoded_label'] = le.fit_transform(label_mapping['Label'])

In [567]:
label_mapping

,Label,ImagePath,encoded_label
0,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R06.jpg,875
1,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R09.jpg,875
2,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R07.jpg,875
3,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R02.jpg,875
4,437-R,DataSet/CASIA-Iris-Thousand/437/R/S5437R03.jpg,875
...,...,...,...
19995,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L04.jpg,1430
19996,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L09.jpg,1430
19997,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L08.jpg,1430
19998,715-L,DataSet/CASIA-Iris-Thousand/715/L/S5715L06.jpg,1430


In [568]:
label_mapping.sort_values(by='encoded_label',inplace=True)
label_mapping

,Label,ImagePath,encoded_label
3331,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L00.jpg,0
3338,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L08.jpg,0
3337,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L07.jpg,0
3336,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L02.jpg,0
3335,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L06.jpg,0
...,...,...,...
14604,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R00.jpg,1999
14603,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R01.jpg,1999
14602,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R03.jpg,1999
14600,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R05.jpg,1999


In [569]:
label_mapping.reset_index(inplace=True)
label_mapping.drop('index',axis=1,inplace=True)

In [570]:
label_mapping.to_csv(r"DataSet\allData.csv")

In [571]:
label_mapping

,Label,ImagePath,encoded_label
0,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L00.jpg,0
1,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L08.jpg,0
2,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L07.jpg,0
3,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L02.jpg,0
4,000-L,DataSet/CASIA-Iris-Thousand/000/L/S5000L06.jpg,0
...,...,...,...
19995,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R00.jpg,1999
19996,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R01.jpg,1999
19997,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R03.jpg,1999
19998,999-R,DataSet/CASIA-Iris-Thousand/999/R/S5999R05.jpg,1999


In [572]:
X=label_mapping.drop('encoded_label',axis=1)
y=label_mapping['encoded_label']

In [573]:
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
X_test, X_val, y_test, y_val = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42,stratify=y_tmp)
len(X_train), len(X_val), len(X_test)

(16000, 2000, 2000)

In [574]:
train_df=pd.concat([X_train, y_train], axis=1)
val_df=pd.concat([X_val, y_val], axis=1)
test_df=pd.concat([X_test, y_test], axis=1)
val_df.sort_values(by='encoded_label',inplace=True)
val_df.reset_index(inplace=True)
val_df.drop('index',axis=1,inplace=True)
train_df.sort_values(by='encoded_label',inplace=True)
train_df.reset_index(inplace=True)
train_df.drop('index',axis=1,inplace=True)
test_df.sort_values(by='encoded_label',inplace=True)    
test_df.reset_index(inplace=True)
test_df.drop('index',axis=1,inplace=True)

In [575]:
train_df.to_csv(r"DataSet\train_data.csv",index=True)
val_df.to_csv(r"DataSet\val_data.csv",index=True)
test_df.to_csv(r"DataSet\test_data.csv",index=True)

In [576]:
class DatasetMaker(Dataset):
    def __init__(self,path,transform=None):
        super().__init__()
        self.map=pd.read_csv(path,index_col=0)
        self.transform=transform
    def __len__(self):
        return len(self.map)
    def __getitem__(self,idx):
        y=self.map['encoded_label'][idx]
        img=Image.open(self.map['ImagePath'][idx]).convert('L')
        if self.transform:
            img=self.transform(img)
        return img,y
        